In [9]:
import os

os.chdir('../')

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [12]:
from MLProject.constant import *
from MLProject.utils.common import read_yaml, create_directories, save_json

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = os.getenv('MLFLOW_TRACKING_URI')
        )
        
        return model_evaluation_config

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import joblib
import mlflow
from urllib.parse import urlparse
import datetime

In [15]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        
        test_x = test_data.drop([self.config.target_column], axis = 1)
        test_y = test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        mlflow.set_experiment('wine-quality-exp')
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run(run_name = f"elasticnet_run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"):
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path = Path(self.config.metric_file_name), data = scores)
            
            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metric('rmse', rmse)
            mlflow.log_metric('mae',mae)
            mlflow.log_metric('r2',r2)
            
            if tracking_url_type_store != 'file':
                mlflow.sklearn.log_model(model, 'model', registered_model_name = 'ElasticnetModel')
            else:
                mlflow.sklearn.log_model(self.model, 'model')
            


In [16]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-17 01:03:54,939: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-17 01:03:54,940: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-17 01:03:54,941: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-17 01:03:54,942: INFO: common: Created directory at: artifacts]
[2026-03-17 01:03:54,942: INFO: common: Created directory at: artifacts/model_evaluation]


/opt/homebrew/Caskroom/miniforge/base/envs/mlproject_end_to_end/lib/python3.8/site-packages/_distutils_hack/__init__.py:32: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/03/17 01:04:02 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetModel, version 6
Created version '6' of model 'ElasticnetModel'.
